<!-- @format -->

# 02 - Data Cleaning & Feature Engineering

Notebook này dành cho làm sạch dữ liệu và tạo feature mới.


<!-- @format -->

## Checklist

- Làm sạch dữ liệu: duplicate, missing, chuẩn hóa text
- Tạo feature thời gian, duration, total stops
- Binary Encoding các cột danh mục quan trọng (Airline, Source, Destination)
- Tạo bản model-ready cho huấn luyện
- Tách train/test để dùng cho bước modeling
- Lưu đầy đủ artifacts vào thư mục `data/processed`


In [1]:
from pathlib import Path

import importlib
import os
import sys

import pandas as pd

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import src.config as config_module
importlib.reload(config_module)

import src.data_preprocessing as dp_module
importlib.reload(dp_module)

import src.feature_engineering as fe_module
importlib.reload(fe_module)

from src.config import (
    CLEAN_FILE,
    FEATURE_FILE,
    MODEL_READY_FILE,
    RAW_FILE,
    TARGET_COL,
    X_TEST_FILE,
    X_TRAIN_FILE,
    Y_TEST_FILE,
    Y_TRAIN_FILE,
)
from src.data_preprocessing import run_preprocessing
from src.feature_engineering import run_step2_pipeline

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print(f"Project root: {project_root}")
print(f"Raw file: {RAW_FILE}")
print(f"Clean file: {CLEAN_FILE}")
print(f"Feature file: {FEATURE_FILE}")
print(f"Model-ready file: {MODEL_READY_FILE}")

Project root: D:\Data_Mining\DataMining-FlightPricePrediction
Raw file: D:\Data_Mining\DataMining-FlightPricePrediction\data\raw\Data_Train.xlsx
Clean file: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\train_clean.csv
Feature file: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\train_features.csv
Model-ready file: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\train_model_ready.csv


In [2]:
raw_df = pd.read_excel(RAW_FILE)
print("Raw shape:", raw_df.shape)

Raw shape: (10683, 11)


In [3]:
clean_df = run_preprocessing(RAW_FILE)
print("Clean shape:", clean_df.shape)
display(clean_df.head())

Clean shape: (10462, 9)


,Airline,Date_of_Journey,Source,Destination,Dep_Time,Arrival_Time,Duration,Total_Stops,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,22:20,01:10 22 Mar,2h 50m,non-stop,3897
1,Air India,1/05/2019,Kolkata,Banglore,05:50,13:15,7h 25m,2 stops,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,09:25,04:25 10 Jun,19h,2 stops,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,18:05,23:30,5h 25m,1 stop,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,16:50,21:35,4h 45m,1 stop,13302


In [4]:
artifacts = run_step2_pipeline()

feature_df = artifacts["feature_df"]
print(feature_df.columns)
model_ready_df = artifacts["model_ready_df"]
print(model_ready_df.columns)
X_train = artifacts["X_train"]
X_test = artifacts["X_test"]
y_train = artifacts["y_train"]
y_test = artifacts["y_test"]

print("Feature shape:", feature_df.shape)
print("Model-ready shape:", model_ready_df.shape)
print("X_train / X_test:", X_train.shape, X_test.shape)
print("y_train / y_test:", y_train.shape, y_test.shape)

Index(['Airline_0', 'Airline_1', 'Airline_2', 'Airline_3', 'Date_of_Journey', 'Source_0', 'Source_1', 'Source_2', 'Destination_0', 'Destination_1',
       'Destination_2', 'Dep_Time', 'Arrival_Time', 'Duration', 'Total_Stops', 'Price', 'journey_day', 'journey_month', 'day_of_week', 'dep_hour',
       'arrival_hour', 'duration_minutes', 'total_stops_num'],
      dtype='object')
Index(['Airline_0', 'Airline_1', 'Airline_2', 'Airline_3', 'Source_0', 'Source_1', 'Source_2', 'Destination_0', 'Destination_1', 'Destination_2', 'Price',
       'journey_day', 'journey_month', 'day_of_week', 'dep_hour', 'arrival_hour', 'duration_minutes', 'total_stops_num'],
      dtype='object')
Feature shape: (10462, 23)
Model-ready shape: (10462, 18)
X_train / X_test: (8369, 17) (2093, 17)
y_train / y_test: (8369,) (2093,)


In [5]:
new_cols = [
    "day_of_week",
    "journey_day",
    "journey_month",
    "dep_hour",
    "dep_minute",
    "arrival_hour",
    "arrival_minute",
    "duration_minutes",
    "total_stops_num"
]

existing_new_cols = [c for c in new_cols if c in feature_df.columns]

print("Các cột feature số mới:")
print(existing_new_cols)

print("\nTỉ lệ missing của các cột feature mới (%):")
display((feature_df[existing_new_cols].isna().mean() * 100).round(2).sort_values(ascending=False))

print("\nThống kê nhanh các cột numeric:")
display(feature_df[existing_new_cols + ([TARGET_COL] if TARGET_COL in feature_df.columns else [])].describe().T)

Các cột feature số mới:
['day_of_week', 'journey_day', 'journey_month', 'dep_hour', 'arrival_hour', 'duration_minutes', 'total_stops_num']

Tỉ lệ missing của các cột feature mới (%):


day_of_week         0.0
journey_day         0.0
journey_month       0.0
dep_hour            0.0
arrival_hour        0.0
duration_minutes    0.0
total_stops_num     0.0
dtype: float64


Thống kê nhanh các cột numeric:


,count,mean,std,min,25%,50%,75%,max
day_of_week,10462.0,2.935576,2.006599,0.0,1.0,3.0,5.00,6.0
journey_day,10462.0,13.463200,8.467493,1.0,6.0,12.0,21.00,27.0
journey_month,10462.0,4.701491,1.163802,3.0,3.0,5.0,6.00,6.0
dep_hour,10462.0,12.478494,5.727227,0.0,8.0,11.0,18.00,23.0
arrival_hour,10462.0,13.387689,6.855547,0.0,8.0,14.0,19.00,23.0
duration_minutes,10462.0,629.781591,500.699045,5.0,170.0,505.0,910.00,2860.0
total_stops_num,10462.0,0.802332,0.660609,0.0,0.0,1.0,1.00,4.0
Price,10462.0,9026.790289,4624.849541,1759.0,5224.0,8266.0,12344.75,79512.0


In [6]:
print("Shape các tập sau khi tách train/test:")
print(f"- X_train: {X_train.shape}")
print(f"- X_test : {X_test.shape}")
print(f"- y_train: {y_train.shape}")
print(f"- y_test : {y_test.shape}")

print("\nTỉ lệ missing trong model_ready_df (%):")
missing_pct = (model_ready_df.isna().mean() * 100).round(2).sort_values(ascending=False)
display(missing_pct.head(15))

print("\nKiểu dữ liệu trong model_ready_df:")
display(model_ready_df.dtypes.value_counts())

Shape các tập sau khi tách train/test:
- X_train: (8369, 17)
- X_test : (2093, 17)
- y_train: (8369,)
- y_test : (2093,)

Tỉ lệ missing trong model_ready_df (%):


Airline_0        0.0
Airline_1        0.0
Airline_2        0.0
Airline_3        0.0
Source_0         0.0
Source_1         0.0
Source_2         0.0
Destination_0    0.0
Destination_1    0.0
Destination_2    0.0
Price            0.0
journey_day      0.0
journey_month    0.0
day_of_week      0.0
dep_hour         0.0
dtype: float64


Kiểu dữ liệu trong model_ready_df:


int64    13
int32     5
Name: count, dtype: int64

In [7]:
print("Saved artifacts:")
print(f"- Clean data: {CLEAN_FILE}")
print(f"- Feature data: {FEATURE_FILE}")
print(f"- Model-ready data: {MODEL_READY_FILE}")
print(f"- X_train: {X_TRAIN_FILE}")
print(f"- X_test: {X_TEST_FILE}")
print(f"- y_train: {Y_TRAIN_FILE}")
print(f"- y_test: {Y_TEST_FILE}")

Saved artifacts:
- Clean data: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\train_clean.csv
- Feature data: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\train_features.csv
- Model-ready data: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\train_model_ready.csv
- X_train: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\X_train.csv
- X_test: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\X_test.csv
- y_train: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\y_train.csv
- y_test: D:\Data_Mining\DataMining-FlightPricePrediction\data\processed\y_test.csv
